# Phase_1

In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import re
import nltk
import pickle
import math
import time
import os

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from collections import defaultdict

# 1. Data Collection:

In [ ]:
import kagglehub

path = kagglehub.dataset_download("thedevastator/220k-gpt4vision-image-captions")
print("Path to dataset files:", path)

In [ ]:
csv_path = None
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_path = os.path.join(root, file)
            break

df = pd.read_csv(csv_path)
print(df.shape)
df.head()

In [ ]:
data = []
for i, row in df.iterrows():
    caption = str(row["caption"]).strip()
    image_url = str(row["url"]).strip()
    if caption:
        data.append({
            "document ID": f"IMG_{i+1}",
            "image_url": image_url,
            "content": caption
        })

df_corpus = pd.DataFrame(data)
df_corpus["length"] = df_corpus["content"].apply(
    lambda x: len(x.split())
)
df_corpus.to_csv(
    "image_search.csv",
    index=False,
    encoding='utf-8-sig'
)
print(f"Total documents processed: {len(df_corpus)}")
print("CSV file created")

df_corpus.head()

# 2. Preprocessing


- Tokenization

- Lowercasing

- Stopword Removal

- Stemming

In [ ]:
nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))
STEMMER = PorterStemmer()
CLEAN_REGEX = re.compile(r'[^a-z\s]+')

In [ ]:
def preprocess(text, remove_stopwords=True, stemming=True):
    text = text.lower().strip()
    text = CLEAN_REGEX.sub(' ', text)
    tokens = text.split()
    if remove_stopwords:
        tokens = [w for w in tokens if w not in STOP_WORDS and len(w) > 2]
    if stemming:
        tokens = [STEMMER.stem(w) for w in tokens]
    return tokens

In [ ]:
for i in range(8):
    print(f"\n Document {i+1}:")
    original_text = df_corpus['content'].iloc[i]
    print("Original:", original_text[:200])
    processed = preprocess(original_text)
    print("Processed:", processed[:20])

# 3. Indexing
An inverted index is created to map each term to the documents containing it along with term frequency.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/image_search.csv")
df["tokens"] = df["content"].apply(preprocess)
print("Data Indexing")
df.head()

In [ ]:
inverted_index = defaultdict(dict)

for _, row in df.iterrows():
    doc_id = row["document ID"]
    tokens = row["tokens"]

    for word in tokens:
        if doc_id not in inverted_index[word]:
            inverted_index[word][doc_id] = 0
        inverted_index[word][doc_id] += 1

print("Inverted Index ")
print("Number of terms:", len(inverted_index))

In [ ]:
sample_terms = list(inverted_index.keys())[:5]

for term in sample_terms:
    print("\nTerm:", term)
    print("Postings sample:", list(inverted_index[term].items())[:10])

# Phase_2

# 6. Query Processing
 Processing the user query and ranking results


In [ ]:
query = "kitchen wooden cabinet"

query_tokens = preprocess(query)
print("Original Query:", query)
print("Processed Query:", query_tokens)

In [ ]:
doc_scores = defaultdict(int)

for term in query_tokens:
    if term in inverted_index:
        postings = inverted_index[term]
        for doc_id, freq in postings.items():
            doc_scores[doc_id] += freq
print("Number of retrieved documents:", len(doc_scores))
print(list(doc_scores.items())[:10])

In [ ]:
N = len(df)

tfidf_scores = defaultdict(float)
for term in query_tokens:
    if term in inverted_index:
        df_term = len(inverted_index[term])
        idf = math.log(N / (df_term + 1))
        for doc_id, tf in inverted_index[term].items():
            tfidf_scores[doc_id] += tf*idf
ranked_docs = sorted(
    tfidf_scores.items(),
    key=lambda x: x[1],
    reverse=True
)
print("Top Ranked Documents:")
for doc_id, score in ranked_docs[:11]:
    print(doc_id, "Score:", round(score, 4))

In [ ]:
top_doc_ids = [doc_id for doc_id, score in ranked_docs[:10]]
results = df[df["document ID"].isin(top_doc_ids)]
results = results.set_index("document ID").loc[top_doc_ids].reset_index()
results[["document ID", "image_url", "content"]]

# 7. User Interface
Building the search engine GUI using Gradio


In [ ]:
import gradio as gr

def image_search_engine(query):
    query_tokens = preprocess(query)
    scores = defaultdict(float)
    for term in query_tokens:
        if term in inverted_index:
            df_term = len(inverted_index[term])
            idf = math.log(N / (df_term + 1))
            for doc_id, tf in inverted_index[term].items():
                scores[doc_id] += tf * idf
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    html = """
    <div style="background-color:white; padding:20px; font-family:Arial;">
    <h1 style="text-align:center; color:black;">Image Search Engine</h1>
    """
    for doc_id, score in ranked[:5]:
        row = df[df["document ID"] == doc_id].iloc[0]
        image_url = row["image_url"].replace("http://", "https://")
        html += f"""
        <div style="
            border:1px solid #ddd;
            background-color:#f9f9f9;
            padding:15px;
            margin:15px;
            border-radius:15px;
        ">
            <h2>{doc_id}</h2>

            <p><b>TF-IDF Score:</b> {round(score, 4)}</p>

            <img src="{image_url}" width="350" style="border-radius:10px;">

            <p>{row['content']}</p>

            <p>
                <a href="{image_url}" target="_blank">Open Image Link</a>
            </p>
        </div>
        """

    html += "</div>"

    return html


interface = gr.Interface(
    fn=image_search_engine,
    inputs=gr.Textbox(
        placeholder="Search images...",
        label="Enter Query"
    ),
    outputs="html",
    title="Search Engine",
    description="Search Engine using TF-IDF Ranking"
)

interface.launch()

# 8. Query Expansion
Expanding search queries using synonyms


In [ ]:
query = "kitchen wooden cabinet"
query_tokens = preprocess(query)
print("Original Query:", query)
print("Processed Query:", query_tokens)

In [ ]:
query = "kitchen wooden cabinet"
query_tokens = preprocess(query)
expansion_dict = {
    "car": ["vehicle", "automobile"],
    "bike": ["bicycle", "motorcycle"],
    "dog": ["puppy", "animal"],
    "cat": ["kitten", "animal"],
    "kitchen": ["stove", "cabinet"],
    "street": ["road", "sidewalk"],
    "person": ["people", "man", "woman"],
    "food": ["meal", "fruit"],
    "bathroom": ["toilet", "sink", "shower"]
}
expanded_query_tokens = query_tokens.copy()
for term in query_tokens:
    if term in expansion_dict:
        expanded_query_tokens.extend(
            expansion_dict[term]
        )
expanded_query_tokens = list(
    set(expanded_query_tokens)
)
print("Original Tokens:", query_tokens)
print("Expanded Tokens:", expanded_query_tokens)

In [ ]:
expanded_scores = defaultdict(float)
for term in expanded_query_tokens:
    if term in inverted_index:
        df_term = len(inverted_index[term])
        idf = math.log(N / (df_term + 1))
        for doc_id, tf in inverted_index[term].items():
            expanded_scores[doc_id] += tf * idf
expanded_ranked_docs = sorted(
    expanded_scores.items(),
    key=lambda x: x[1],
    reverse=True
)
print("Top Results After Query Expansion:")
for doc_id, score in expanded_ranked_docs[:10]:
    print(doc_id, "Score:", round(score, 4))

# 9. Evaluation
Measuring retrieval performance and speed


In [ ]:
test_queries = [
    "kitchen cabinet",
    "bathroom sink",
    "motorcycle",
    "street bicycle",
    "cat cabinet"
]

In [ ]:
def search_query(query):
    query_tokens = preprocess(query)
    scores = defaultdict(float)
    for term in query_tokens:
        if term in inverted_index:
            df_term = len(inverted_index[term])
            idf = math.log(N / (df_term + 1))
            for doc_id, tf in inverted_index[term].items():
                scores[doc_id] += tf * idf
    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    return ranked

In [ ]:
for q in test_queries:
    start_time = time.time()
    ranked = search_query(q)
    end_time = time.time()
    print("\nQuery:", q)
    print("Retrieved Documents:", len(ranked))
    print("Search Time:", round(end_time - start_time, 4), "seconds")
    if ranked:
        print("Top Result:", ranked[0])

# 10. Saving and Loading Index
Saving the inverted index for reuse


In [ ]:
with open("inverted_index.pkl", "wb") as f:
    pickle.dump(dict(inverted_index), f)
print("Index Saved ")

In [ ]:
with open("inverted_index.pkl", "rb") as f:
    loaded_index = pickle.load(f)
print("Index Loaded")

In [ ]:
print("Total Documents:", len(df))
print("Vocabulary Size:", len(inverted_index))
total_tokens = sum(len(tokens) for tokens in df["tokens"])
print("Total Tokens:", total_tokens)